# Week 11 Lab — AutoML with AutoGluon, H2O, and Leaderboard Interpretation

## 40분 실습 목표
1. `scikit-learn` baseline을 먼저 만들어 AutoML의 비교 기준을 세운다.
2. AutoML이 자동화하는 작업을 `leaderboard` 관점에서 확인한다.
3. AutoGluon이 설치되어 있으면 짧은 코드로 Tabular AutoML을 실행한다.
4. H2O가 설치되어 있으면 H2O AutoML leaderboard도 비교한다.
5. 설치가 어려운 환경에서도 `AutoML-lite`로 동일한 모델 선택 사고방식을 연습한다.

---

## 오늘의 핵심 질문

> AutoML은 “모델링을 안 해도 되는 도구”가 아니라, 반복 탐색과 비교를 자동화해 baseline 확보 속도를 높이는 도구입니다.  
> 따라서 오늘 실습의 핵심은 **leaderboard를 보고 어떤 모델을 선택할지 설명하는 것**입니다.

## 권장 시간표
- 0~5분: 데이터와 baseline 확인
- 5~15분: AutoML-lite leaderboard 만들기
- 15~25분: AutoGluon 실습
- 25~33분: H2O AutoML 선택 실습
- 33~40분: leaderboard 해석과 운영 관점 선택

> **발표자 스크립트**  
> "이번 실습에서는 AutoML을 라이브러리 이름으로만 배우지 않겠습니다. 먼저 우리가 직접 만든 baseline을 기준점으로 삼고, AutoML이 어떤 후보들을 비교하는지 leaderboard로 확인하겠습니다. 중요한 것은 1등 모델 이름을 외우는 것이 아니라, 시간 예산, 성능, 해석 가능성, 배포 가능성까지 고려해 선택하는 습관입니다."

## 0. 실습 전 준비

필수 패키지는 `pandas`, `numpy`, `scikit-learn`입니다.  
AutoGluon과 H2O는 설치 시간이 길 수 있으므로, 수업 환경에 이미 설치되어 있으면 실행하고 아니면 건너뛰도록 구성했습니다.

> 중요: 노트북에서 `!uv add ...`를 실행하는 방식은 권장하지 않습니다.  
> `uv add`는 프로젝트의 `pyproject.toml`과 `.venv`를 수정하는 명령이고, Jupyter 커널이 사용하는 Python과 다를 수 있습니다.  
> 노트북 안에서 급하게 설치해야 한다면 현재 커널에 직접 설치되는 `%pip install ...`을 사용하세요.


## 0-1. Windows / macOS 공통 환경 원칙

수업에서 제일 많이 나는 오류는 패키지가 없는 것이 아니라, **패키지를 설치한 Python과 노트북 커널 Python이 다른 것**입니다.

반드시 아래 순서를 지켜주세요.

1. 저장소 루트로 이동한다.
2. `uv sync`로 프로젝트 `.venv`를 만든다.
3. Jupyter를 `uv run jupyter lab`로 실행한다.
4. 노트북 커널을 프로젝트 `.venv`로 선택한다.
5. 패키지가 없다고 나오면 노트북 안에서는 `%pip install ...`을 사용한다.

### macOS zsh 기준

```bash
cd ~/smilechacha/ajou-2026-1st-sem/ajou-mlops-2026-1nd-semester
uv sync
uv run python -m ipykernel install --user --name ajou-mlops-2026 --display-name "Ajou MLOps 2026 (.venv)"
uv run jupyter lab
```

H2O까지 실행하려면 Java가 필요합니다.

```bash
brew install openjdk@17
export JAVA_HOME="/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
export PATH="/opt/homebrew/opt/openjdk@17/bin:$PATH"
uv sync --extra automl
```

### Windows PowerShell 기준

```powershell
cd C:\path\to\ajou-mlops-2026-1nd-semester
uv sync
uv run python -m ipykernel install --user --name ajou-mlops-2026 --display-name "Ajou MLOps 2026 (.venv)"
uv run jupyter lab
```

Windows에서 H2O까지 실행하려면 Java 17이 필요합니다. 아래 중 하나를 사용하세요.

```powershell
winget install EclipseAdoptium.Temurin.17.JDK
```

설치 후 새 PowerShell을 열고 확인합니다.

```powershell
java -version
uv sync --extra automl
```

### Colab / 일반 Jupyter 기준

Colab이나 학교 PC처럼 `uv` 프로젝트 환경을 쓰기 어려운 경우에는 노트북 안에서 아래처럼 현재 커널에 직접 설치합니다.

```python
%pip install pandas numpy scikit-learn matplotlib seaborn
%pip install autogluon.tabular h2o
```

AutoGluon/H2O 설치가 오래 걸리면 해당 셀은 건너뛰고 `AutoML-lite` 실습만 진행해도 됩니다.

In [1]:
# 현재 노트북 커널/환경 진단 셀
# 문제가 생기면 이 셀 출력부터 확인합니다.

import os
import sys
import shutil
import platform
import subprocess
import importlib.util
from pathlib import Path

print("OS:", platform.platform())
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Current working directory:", Path.cwd())
print("VIRTUAL_ENV:", os.environ.get("VIRTUAL_ENV"))
print("uv path:", shutil.which("uv"))
print("java path:", shutil.which("java"))
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

for package, import_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scikit-learn", "sklearn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("autogluon.tabular", "autogluon.tabular"),
    ("h2o", "h2o"),
]:
    found = importlib.util.find_spec(import_name) is not None
    print(f"{package:18s}:", "OK" if found else "missing")

if shutil.which("java"):
    try:
        first_line = subprocess.check_output(
            ["java", "-version"], stderr=subprocess.STDOUT, text=True
        ).splitlines()[0]
        print("Java version:", first_line)
    except Exception as e:
        print("Java check failed:", repr(e))

OS: macOS-26.3.1-arm64-arm-64bit
Python executable: /Users/musinsa/smilechacha/ajou-2026-1st-sem/.venv/bin/python
Python version: 3.12.13
Current working directory: /Users/musinsa/smilechacha/ajou-2026-1st-sem/ajou-mlops-2026-1nd-semester
VIRTUAL_ENV: /Users/musinsa/smilechacha/ajou-2026-1st-sem/ajou-mlops-2026-1nd-semester/.venv
uv path: /Users/musinsa/smilechacha/ajou-2026-1st-sem/.venv/bin/uv
java path: /opt/homebrew/opt/openjdk@17/bin/java
JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
numpy             : OK
pandas            : OK
scikit-learn      : OK
matplotlib        : OK
seaborn           : OK
autogluon.tabular : OK
h2o               : OK
Java version: openjdk version "17.0.19" 2026-04-21


In [ ]:
# 필요 패키지 설치 예시
# 노트북에서는 현재 선택된 커널에 직접 설치되는 %pip 방식을 권장합니다.
# !uv add는 프로젝트 pyproject.toml을 수정하는 명령이라 커널 환경과 어긋날 수 있습니다.

# 기본 실습 패키지: import 오류가 날 때만 주석을 풀고 실행하세요.
# %pip install pandas numpy scikit-learn matplotlib seaborn

# 선택 실습 패키지: 설치 시간이 오래 걸릴 수 있습니다.
# %pip install autogluon.tabular h2o

# macOS / Windows 로컬 uv 프로젝트 환경을 동기화하고 싶다면 노트북이 아니라 터미널에서 실행하세요.
# uv sync
# uv sync --extra automl

# H2O는 Java 17이 필요합니다.
# macOS: brew install openjdk@17
# Windows: winget install EclipseAdoptium.Temurin.17.JDK

# 설치 시간이 오래 걸리면 AutoGluon/H2O 셀은 건너뛰고 AutoML-lite 셀만 실행해도 됩니다.


In [2]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 1. 데이터 로드

이번 실습은 인터넷 연결 없이도 실행 가능한 `sklearn.datasets.load_breast_cancer` 데이터를 사용합니다.

- 목적: 이진 분류
- target: `malignant` / `benign`
- 장점: 데이터가 작아서 40분 실습 안에 여러 모델을 빠르게 비교할 수 있음

In [3]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

df = X.copy()
df["target"] = y
df["target_name"] = df["target"].map({0: "malignant", 1: "benign"})

print(df.shape)
df.head()

(569, 32)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target,target_name
0,17.9900,10.3800,122.8000,1001.0000,0.1184,0.2776,0.3001,0.1471,0.2419,0.0787,1.0950,0.9053,8.5890,153.4000,0.0064,0.0490,0.0537,0.0159,0.0300,0.0062,25.3800,17.3300,184.6000,2019.0000,0.1622,0.6656,0.7119,0.2654,0.4601,0.1189,0,malignant
1,20.5700,17.7700,132.9000,1326.0000,0.0847,0.0786,0.0869,0.0702,0.1812,0.0567,0.5435,0.7339,3.3980,74.0800,0.0052,0.0131,0.0186,0.0134,0.0139,0.0035,24.9900,23.4100,158.8000,1956.0000,0.1238,0.1866,0.2416,0.1860,0.2750,0.0890,0,malignant
2,19.6900,21.2500,130.0000,1203.0000,0.1096,0.1599,0.1974,0.1279,0.2069,0.0600,0.7456,0.7869,4.5850,94.0300,0.0062,0.0401,0.0383,0.0206,0.0225,0.0046,23.5700,25.5300,152.5000,1709.0000,0.1444,0.4245,0.4504,0.2430,0.3613,0.0876,0,malignant
3,11.4200,20.3800,77.5800,386.1000,0.1425,0.2839,0.2414,0.1052,0.2597,0.0974,0.4956,1.1560,3.4450,27.2300,0.0091,0.0746,0.0566,0.0187,0.0596,0.0092,14.9100,26.5000,98.8700,567.7000,0.2098,0.8663,0.6869,0.2575,0.6638,0.1730,0,malignant
4,20.2900,14.3400,135.1000,1297.0000,0.1003,0.1328,0.1980,0.1043,0.1809,0.0588,0.7572,0.7813,5.4380,94.4400,0.0115,0.0246,0.0569,0.0188,0.0176,0.0051,22.5400,16.6700,152.2000,1575.0000,0.1374,0.2050,0.4000,0.1625,0.2364,0.0768,0,malignant


In [4]:
print("Target distribution")
display(df["target_name"].value_counts(normalize=True).rename("ratio").to_frame())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("train:", X_train.shape, "test:", X_test.shape)

Target distribution


,ratio
target_name,
benign,0.6274
malignant,0.3726


train: (455, 30) test: (114, 30)


## 2. Manual baseline 만들기

AutoML 결과가 좋아 보이려면 비교 기준이 필요합니다.  
먼저 사람이 직접 만든 간단한 baseline을 만듭니다.

In [5]:
baseline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]
)

start = time.time()
baseline.fit(X_train, y_train)
fit_time = time.time() - start

pred = baseline.predict(X_test)
proba = baseline.predict_proba(X_test)[:, 1]

baseline_result = {
    "model": "Manual LogisticRegression",
    "accuracy": accuracy_score(y_test, pred),
    "auc": roc_auc_score(y_test, proba),
    "f1": f1_score(y_test, pred),
    "fit_time_sec": fit_time,
}

pd.DataFrame([baseline_result])

,model,accuracy,auc,f1,fit_time_sec
0,Manual LogisticRegression,0.9825,0.9954,0.9861,0.0069


> **해석 포인트**  
> Baseline은 반드시 복잡할 필요가 없습니다. 빠르게 만들 수 있고, 재현 가능하며, 이후 모델이 정말 좋아졌는지 비교할 수 있으면 충분합니다.

## 3. AutoML-lite: 직접 만드는 작은 leaderboard

AutoGluon/H2O를 쓰기 전에, AutoML의 핵심 아이디어를 작은 코드로 흉내내 봅니다.

자동화 대상:
- 모델 후보 여러 개 학습
- 동일 metric으로 비교
- 학습 시간 기록
- leaderboard 정렬

In [6]:
candidate_models = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        random_state=RANDOM_STATE,
    ),
}

rows = []
trained_models = {}

for name, model in candidate_models.items():
    start = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start

    pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)[:, 1]
    else:
        proba = pred

    rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, pred),
            "auc": roc_auc_score(y_test, proba),
            "f1": f1_score(y_test, pred),
            "fit_time_sec": fit_time,
        }
    )
    trained_models[name] = model

leaderboard_lite = (
    pd.DataFrame(rows)
    .sort_values(["auc", "accuracy", "f1"], ascending=False)
    .reset_index(drop=True)
)

leaderboard_lite

,model,accuracy,auc,f1,fit_time_sec
0,LogisticRegression,0.9825,0.9954,0.9861,0.0039
1,RandomForest,0.9474,0.9937,0.9583,0.1972
2,ExtraTrees,0.9561,0.9932,0.9655,0.1233
3,HistGradientBoosting,0.9737,0.9927,0.9796,1.8456


In [7]:
best_lite_name = leaderboard_lite.loc[0, "model"]
best_lite_model = trained_models[best_lite_name]

print("Best AutoML-lite model:", best_lite_name)
display(leaderboard_lite.head())

Best AutoML-lite model: LogisticRegression


,model,accuracy,auc,f1,fit_time_sec
0,LogisticRegression,0.9825,0.9954,0.9861,0.0039
1,RandomForest,0.9474,0.9937,0.9583,0.1972
2,ExtraTrees,0.9561,0.9932,0.9655,0.1233
3,HistGradientBoosting,0.9737,0.9927,0.9796,1.8456


## 4. Leaderboard 해석 연습

아래 질문에 답해봅니다.

1. AUC 1등 모델이 항상 배포하기 좋은 모델인가?
2. baseline 대비 성능 차이가 충분히 큰가?
3. 학습 시간이 길어져도 감수할 가치가 있는가?
4. 다음 주 XAI 관점에서 설명 가능한 모델인가?

In [8]:
comparison = pd.concat(
    [
        pd.DataFrame([baseline_result]),
        leaderboard_lite.assign(model=lambda d: "AutoML-lite " + d["model"]),
    ],
    ignore_index=True,
).sort_values("auc", ascending=False)

comparison

,model,accuracy,auc,f1,fit_time_sec
0,Manual LogisticRegression,0.9825,0.9954,0.9861,0.0069
1,AutoML-lite LogisticRegression,0.9825,0.9954,0.9861,0.0039
2,AutoML-lite RandomForest,0.9474,0.9937,0.9583,0.1972
3,AutoML-lite ExtraTrees,0.9561,0.9932,0.9655,0.1233
4,AutoML-lite HistGradientBoosting,0.9737,0.9927,0.9796,1.8456


## 5. AutoGluon Tabular 실습

AutoGluon이 설치되어 있으면 아래 셀을 실행합니다.  
설치되어 있지 않으면 셀이 자동으로 안내 메시지를 출력하고 넘어갑니다.

AutoGluon의 장점:
- 짧은 코드로 여러 모델과 앙상블을 자동 비교
- `leaderboard()`로 후보 모델의 성능을 확인
- tabular 데이터 baseline 확보에 강함

In [9]:
try:
    from autogluon.tabular import TabularPredictor
    AUTOGLUON_AVAILABLE = True
except Exception as e:
    AUTOGLUON_AVAILABLE = False
    print("AutoGluon is not available in this environment.")
    print("Skip this section or install with: pip install autogluon.tabular")
    print("Import error:", repr(e))

In [10]:
if AUTOGLUON_AVAILABLE:
    train_ag = X_train.copy()
    train_ag["target"] = y_train.values

    test_ag = X_test.copy()
    test_ag["target"] = y_test.values

    predictor = TabularPredictor(
        label="target",
        eval_metric="roc_auc",
        problem_type="binary",
        path="AutogluonModels/week11_automl",
        verbosity=2,
    )

    predictor.fit(
        train_data=train_ag,
        time_limit=120,
        presets="medium_quality",
    )

    ag_leaderboard = predictor.leaderboard(test_ag, silent=True)
    display(ag_leaderboard.head(10))
else:
    ag_leaderboard = None

Verbosity: 2 (Standard Logging)
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.3.0: Wed Jan 28 20:51:28 PST 2026; root:xnu-12377.91.3~2/RELEASE_ARM64_T6041
CPU Count:          14
Pytorch Version:    Can't import torch
CUDA Version:       Can't get cuda version from torch
Memory Avail:       10.77 GB / 48.00 GB (22.4%)
Disk Space Avail:   297.98 GB / 460.43 GB (64.7%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "/Users/musinsa/smilechacha/ajou-2026-1st-sem/ajou-mlops-2026-1nd-semester/AutogluonModels/week11_automl"
Train Data Rows:    455
Train Data Columns:

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,ExtraTreesGini,0.9944,0.9933,roc_auc,0.0180,0.0137,0.1637,0.0180,0.0137,0.1637,1,True,3
1,WeightedEnsemble_L2,0.9937,0.9943,roc_auc,0.0347,0.0276,0.3526,0.0013,0.0006,0.0064,2,True,5
2,RandomForestEntr,0.9934,0.9925,roc_auc,0.0153,0.0132,0.1826,0.0153,0.0132,0.1826,1,True,2
3,ExtraTreesEntr,0.9921,0.9923,roc_auc,0.0177,0.0272,0.1855,0.0177,0.0272,0.1855,1,True,4
4,RandomForestGini,0.9907,0.9920,roc_auc,0.0167,0.0146,0.1827,0.0167,0.0146,0.1827,1,True,1


In [11]:
if AUTOGLUON_AVAILABLE:
    ag_pred = predictor.predict(test_ag.drop(columns=["target"]))
    ag_proba = predictor.predict_proba(test_ag.drop(columns=["target"]))[1]

    ag_result = {
        "model": "AutoGluon best",
        "accuracy": accuracy_score(y_test, ag_pred),
        "auc": roc_auc_score(y_test, ag_proba),
        "f1": f1_score(y_test, ag_pred),
        "fit_time_sec": np.nan,
    }

    display(pd.DataFrame([ag_result]))
else:
    ag_result = None

,model,accuracy,auc,f1,fit_time_sec
0,AutoGluon best,0.9561,0.9937,0.9655,NaN


> **AutoGluon 해석 포인트**  
> `leaderboard`에서 `score_test`만 보지 말고 `fit_time`, `pred_time_test`, `model` 이름을 함께 봅니다.  
> 수업에서는 “가장 높은 점수”가 아니라 “왜 그 모델을 선택했는가”를 설명하는 것이 목표입니다.

## 6. H2O AutoML 선택 실습

H2O가 설치되어 있으면 아래 셀을 실행합니다.  
H2O는 별도의 Java 기반 런타임을 사용하므로 실행 환경에 따라 초기화 시간이 걸릴 수 있습니다.

### H2O 실행 조건

- Python package: `h2o`
- Java Runtime: Java 17 권장
- macOS 설치 예시: `brew install openjdk@17`
- Windows 설치 예시: `winget install EclipseAdoptium.Temurin.17.JDK`

Java 또는 H2O 서버 시작이 실패하면 이 노트북은 H2O 섹션만 건너뛰고 계속 진행하도록 작성되어 있습니다.


In [12]:
import os
import shutil
import subprocess

try:
    import h2o
    from h2o.automl import H2OAutoML
    H2O_AVAILABLE = True
except Exception as e:
    H2O_AVAILABLE = False
    print("H2O Python package is not available in this environment.")
    print("Skip this section or install with: %pip install h2o")
    print("Import error:", repr(e))

JAVA_HOME_CANDIDATES = [
    os.environ.get("JAVA_HOME"),
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
]

for java_home in JAVA_HOME_CANDIDATES:
    if java_home and os.path.exists(os.path.join(java_home, "bin", "java")):
        os.environ["JAVA_HOME"] = java_home
        os.environ["PATH"] = os.path.join(java_home, "bin") + os.pathsep + os.environ.get("PATH", "")
        break

JAVA_AVAILABLE = shutil.which("java") is not None
if JAVA_AVAILABLE:
    print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT, text=True).splitlines()[0])
else:
    print("Java runtime is not available. H2O AutoML section will be skipped.")
    print("On macOS, install Java with: brew install openjdk@17")
    H2O_AVAILABLE = False


openjdk version "17.0.19" 2026-04-21


In [13]:
h2o_leaderboard = None
aml = None
test_hf = None

if H2O_AVAILABLE:
    try:
        h2o.init(max_mem_size="2G", port=54321)

        train_h2o = X_train.copy()
        train_h2o["target"] = y_train.astype(str).values

        test_h2o = X_test.copy()
        test_h2o["target"] = y_test.astype(str).values

        train_hf = h2o.H2OFrame(train_h2o)
        test_hf = h2o.H2OFrame(test_h2o)
        train_hf["target"] = train_hf["target"].asfactor()
        test_hf["target"] = test_hf["target"].asfactor()

        aml = H2OAutoML(
            max_runtime_secs=120,
            seed=RANDOM_STATE,
            sort_metric="AUC",
            nfolds=3,
        )
        aml.train(
            x=list(X_train.columns),
            y="target",
            training_frame=train_hf,
        )

        h2o_leaderboard = aml.leaderboard.as_data_frame()
        display(h2o_leaderboard.head(10))
    except Exception as e:
        H2O_AVAILABLE = False
        print("H2O AutoML could not start in this environment; skipping H2O section.")
        print("Reason:", repr(e))
else:
    print("H2O section skipped. Continue with AutoML-lite and AutoGluon sections.")


Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.19" 2026-04-21; OpenJDK Runtime Environment Homebrew (build 17.0.19+0); OpenJDK 64-Bit Server VM Homebrew (build 17.0.19+0, mixed mode, sharing)
  Starting server from /Users/musinsa/smilechacha/ajou-2026-1st-sem/ajou-mlops-2026-1nd-semester/.venv/lib/python3.12/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /var/folders/1c/9r08y0pd1b15z2ylxfr_l6p80000gn/T/tmpb2ezw9c6
  JVM stdout: /var/folders/1c/9r08y0pd1b15z2ylxfr_l6p80000gn/T/tmpb2ezw9c6/h2o_musinsa_started_from_python.out
  JVM stderr: /var/folders/1c/9r08y0pd1b15z2ylxfr_l6p80000gn/T/tmpb2ezw9c6/h2o_musinsa_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,Asia/Seoul
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,1 month and 28 days
H2O_cluster_name:,H2O_from_python_musinsa_00qexp
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,2 Gb
H2O_cluster_total_cores:,14
H2O_cluster_allowed_cores:,14
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
20:40:12.738: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


/Users/musinsa/smilechacha/ajou-2026-1st-sem/.venv/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


,model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
0,StackedEnsemble_BestOfFamily_4_AutoML_1_202605...,0.9967,0.0590,0.9977,0.0194,0.1223,0.0149
1,DeepLearning_grid_1_AutoML_1_20260510_204012_m...,0.9965,0.0818,0.9979,0.0235,0.1398,0.0195
2,DeepLearning_grid_2_AutoML_1_20260510_204012_m...,0.9964,0.0868,0.9976,0.0165,0.1209,0.0146
3,StackedEnsemble_BestOfFamily_6_AutoML_1_202605...,0.9963,0.0613,0.9974,0.0182,0.1259,0.0159
4,StackedEnsemble_AllModels_3_AutoML_1_20260510_...,0.9954,0.1300,0.9966,0.0176,0.1537,0.0236
5,GBM_grid_1_AutoML_1_20260510_204012_model_59,0.9949,0.0842,0.9968,0.0270,0.1539,0.0237
6,StackedEnsemble_BestOfFamily_3_AutoML_1_202605...,0.9949,0.0726,0.9965,0.0200,0.1356,0.0184
7,DeepLearning_grid_1_AutoML_1_20260510_204012_m...,0.9948,0.0929,0.9967,0.0264,0.1527,0.0233
8,GBM_grid_1_AutoML_1_20260510_204012_model_82,0.9947,0.0816,0.9965,0.0329,0.1523,0.0232
9,DeepLearning_grid_1_AutoML_1_20260510_204012_m...,0.9947,0.1181,0.9967,0.0347,0.1626,0.0264


In [14]:
if H2O_AVAILABLE and aml is not None and test_hf is not None:
    perf = aml.leader.model_performance(test_hf)
    h2o_result = {
        "model": "H2O AutoML leader",
        "accuracy": np.nan,
        "auc": perf.auc(),
        "f1": np.nan,
        "fit_time_sec": np.nan,
    }
    display(pd.DataFrame([h2o_result]))
else:
    h2o_result = None


,model,accuracy,auc,f1,fit_time_sec
0,H2O AutoML leader,NaN,0.9940,NaN,NaN


## 7. 최종 비교 테이블 만들기

실행 가능한 도구 결과만 모아 최종 비교 테이블을 만듭니다.

In [15]:
final_rows = [baseline_result]
final_rows.extend(leaderboard_lite.to_dict("records"))

if ag_result is not None:
    final_rows.append(ag_result)

if h2o_result is not None:
    final_rows.append(h2o_result)

final_comparison = (
    pd.DataFrame(final_rows)
    .sort_values("auc", ascending=False)
    .reset_index(drop=True)
)

final_comparison

,model,accuracy,auc,f1,fit_time_sec
0,Manual LogisticRegression,0.9825,0.9954,0.9861,0.0069
1,LogisticRegression,0.9825,0.9954,0.9861,0.0039
2,H2O AutoML leader,NaN,0.9940,NaN,NaN
3,RandomForest,0.9474,0.9937,0.9583,0.1972
4,AutoGluon best,0.9561,0.9937,0.9655,NaN
5,ExtraTrees,0.9561,0.9932,0.9655,0.1233
6,HistGradientBoosting,0.9737,0.9927,0.9796,1.8456


## 8. 해석 가능성 맛보기: Permutation Importance

다음 주 XAI와 연결하기 위해, 오늘 선택한 모델의 feature importance를 간단히 확인합니다.  
여기서는 AutoML-lite best 모델을 대상으로 permutation importance를 계산합니다.

In [16]:
perm = permutation_importance(
    best_lite_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
)

importance = (
    pd.DataFrame(
        {
            "feature": X_test.columns,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .head(10)
)

importance

,feature,importance_mean,importance_std
24,worst smoothness,0.0035,0.0017
13,area error,0.0034,0.0034
10,radius error,0.0034,0.0024
21,worst texture,0.0034,0.0030
27,worst concave points,0.0032,0.0025
23,worst area,0.0032,0.0028
20,worst radius,0.0030,0.0031
7,mean concave points,0.0021,0.0021
22,worst perimeter,0.0021,0.0022
26,worst concavity,0.0016,0.0018


## 9. 실습 정리 질문

아래 질문에 2~3문장으로 답해봅니다.

1. 오늘의 best model은 무엇인가?
2. 그 모델을 선택한 이유는 성능 때문인가, 시간 때문인가, 해석 가능성 때문인가?
3. AutoML이 자동화해준 부분은 무엇인가?
4. AutoML이 자동화하지 못한 부분은 무엇인가?
5. 다음 주 XAI에서 어떤 질문을 이어가야 하는가?

In [17]:
my_decision = {
    "selected_model": "",
    "reason": "",
    "risk_or_limit": "",
    "next_xai_question": "",
}

my_decision

{'selected_model': '',
 'reason': '',
 'risk_or_limit': '',
 'next_xai_question': ''}

## 10. 마무리

오늘 실습에서 확인한 것:

- AutoML의 핵심은 모델 하나가 아니라 **후보 탐색과 비교 workflow**이다.
- Leaderboard는 최종 답안지가 아니라 **의사결정 자료**이다.
- 성능이 가장 좋은 모델이 항상 운영에 가장 좋은 모델은 아니다.
- 다음 주 XAI에서는 “왜 이 모델이 이런 예측을 했는가?”로 질문이 이어진다.